In [1]:
import os
import shutil
import subprocess
from pathlib import Path

DEST = Path("/kaggle/working/Nemotron-training")
# Public repo (default). Private repo: set NEMOTRON_GIT_URL to
# "https://<YOUR_TOKEN>@github.com/bayntun/Nemotron-training.git"
URL = os.environ.get(
    "NEMOTRON_GIT_URL",
    "https://github.com/bayntun/Nemotron-training.git",
)

DEST.parent.mkdir(parents=True, exist_ok=True)
if DEST.exists():
    shutil.rmtree(DEST, ignore_errors=True)

env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
print("Cloning", URL, "->", DEST)
r = subprocess.run(
    ["git", "clone", "--depth", "1", URL, str(DEST)],
    capture_output=True,
    text=True,
    env=env,
)
print(r.stderr or r.stdout or "(no output)")
assert r.returncode == 0, "git clone failed — check Internet and URL"
assert (DEST / "eval" / "test_grader.py").is_file(), "clone incomplete"
print("OK:", DEST)

Cloning https://github.com/bayntun/Nemotron-training.git -> /kaggle/working/Nemotron-training
Cloning into '/kaggle/working/Nemotron-training'...

OK: /kaggle/working/Nemotron-training


In [2]:
# VERIFY CLONE — run after the clone cell in this same notebook session.

from __future__ import annotations

import os
import subprocess
from pathlib import Path

MARKER = Path("eval") / "test_grader.py"
CANDIDATES = [
    Path("/kaggle/working/Nemotron-training"),
    Path("/kaggle/working/Nemotron"),
    Path.cwd(),
]


def check(label: str, base: Path) -> bool:
    base = base.resolve()
    marker = base / MARKER
    ok = marker.is_file()
    print(f"{label}: {base}")
    print(f"  exists: {base.is_dir()}  has {MARKER}: {ok}")
    return ok


print("Python cwd:", Path.cwd())
print("-" * 72)

found: list[Path] = []
for lbl, p in [
    ("default clone path", CANDIDATES[0]),
    ("alt name", CANDIDATES[1]),
    ("cwd", CANDIDATES[2]),
]:
    if check(lbl, p):
        found.append(p.resolve())

if found:
    root = found[0]
    print("-" * 72)
    print("OK: repo root looks like:", root)
    for nm in ("README.md", "pyproject.toml", "requirements.txt"):
        fp = root / nm
        print(f"  {nm}: {fp.is_file()}")
    git = subprocess.run(
        ["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
        capture_output=True,
        text=True,
    )
    print("  git HEAD:", (git.stdout or git.stderr).strip() or "(not a git checkout?)")
else:
    print("-" * 72)
    print("No repo found at the usual paths above.")
    print("Listing /kaggle/working:")
    w = Path("/kaggle/working")
    if w.is_dir():
        for ch in sorted(w.iterdir())[:40]:
            print(" ", ch)
        if sum(1 for _ in w.iterdir()) > 40:
            print("  ...")
    else:
        print("  (missing /kaggle/working)")

Python cwd: /kaggle/working
------------------------------------------------------------------------
default clone path: /kaggle/working/Nemotron-training
  exists: True  has eval/test_grader.py: True
alt name: /kaggle/working/Nemotron
  exists: False  has eval/test_grader.py: False
cwd: /kaggle/working
  exists: True  has eval/test_grader.py: False
------------------------------------------------------------------------
OK: repo root looks like: /kaggle/working/Nemotron-training
  README.md: True
  pyproject.toml: True
  requirements.txt: True
  git HEAD: 9113c21


In [ ]:
from kaggle_secrets import UserSecretsClient
u = UserSecretsClient()
print("HF_TOKEN:", bool(u.get_secret("HF_TOKEN")))
print("DEEPSEEK_API_KEY:", bool(u.get_secret("DEEPSEEK_API_KEY")))

In [1]:
!nvidia-smi

Thu May  7 09:35:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----